# 配置和自定义chain
通常情况下，您可能希望尝试甚至向最终用户展示多种不同的做事方式。为了使这种体验尽可能简单，我们定义了两个方法。

首先，`configurable_fields `允许您配置可运行文件的特定字段。

第二，`configurable_alternatives `方法，您可以列出在运行时期间可以设置的任何特定可运行程序的替代方案。

In [14]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from langchain.globals import set_debug
set_debug(False) 

## Configuration Fields
### 使用LLMs进行配置
在LLMs中可以直接配置`temperature`等字段：

In [16]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import ConfigurableField






In [17]:
model = ChatOpenAI(temperature=0)
model

ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x00000208C0728550>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x00000208C0729E10>, temperature=0.0, openai_api_key=SecretStr('**********'), openai_api_base='https://flag.smarttrot.com/v1', openai_proxy='')

In [18]:
model = ChatOpenAI(temperature=0).configurable_fields(
    temperature=ConfigurableField(
        id="llm_temperature",
        name="LLM Temperature",
        description="The temperature of the LLM",
    )
)
model

RunnableConfigurableFields(default=ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x00000208C072BC70>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x00000208C073DAB0>, temperature=0.0, openai_api_key=SecretStr('**********'), openai_api_base='https://flag.smarttrot.com/v1', openai_proxy=''), fields={'temperature': ConfigurableField(id='llm_temperature', name='LLM Temperature', description='The temperature of the LLM', annotation=None, is_shared=False)})

In [ ]:
model.invoke("pick a random number")  # 输出：AIMessage(content='7')

In [ ]:
model.with_config(configurable={"llm_temperature": 0.9}).invoke("pick a random number")


我们也可以在一个chain中进行配置：

In [ ]:
prompt = PromptTemplate.from_template("Pick a random number above {x}")
chain = prompt | model

chain.invoke({"x": 0}) # 输出：AIMessage(content='57')


## 使用HubRunnables进行配置
[HubRunnable](https://api.python.langchain.com/en/stable/runnables/langchain.runnables.hub.HubRunnable.html#langchain.runnables.hub.HubRunnable)是`LangChain`提供的一个可运行组件，作用是可以拉取`GitHub`上公开的`prompt`文件使用。下面我们使用`HubRunnable("rlm/rag-prompt")`:新建了一个`HubRunnable`实例，指定去拉取`"rlm/rag-prompt"`这个`repo`中的默认`prompt`文件。


In [5]:
from langchain.runnables.hub import HubRunnable

prompt = HubRunnable("rlm/rag-prompt").configurable_fields(
    owner_repo_commit=ConfigurableField(
        id="hub_commit",
        name="Hub Commit",
        description="The Hub commit to pull from",
    )
)
prompt.input_schema()


PromptInput(context=None, question=None)

In [10]:
prompt

RunnableConfigurableFields(default=HubRunnable(bound=ChatPromptTemplate(input_variables=['context', 'question'], metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"))]), owner_repo_commit='rlm/rag-prompt'), fields={'owner_repo_commit': ConfigurableField(id='hub_commit', name='Hub Commit', description='The Hub commit to pull from', annotation=None, is_shared=False)})

 我们使用`configurable_fields`方法定义了`HubRunnable`中可配置的字段为`owner_repo_commit`，使用`ConfigurableField（id）`定义这个字段具体的可配置的元信息为字段ID、名称、描述。然后使用默认配置通过`prompt.invoke()`方法调用该`Runnable`，生成`prompt`。



In [6]:
prompt.invoke({"question": "foo", "context": "bar"})

ChatPromptValue(messages=[HumanMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: foo \nContext: bar \nAnswer:")])

下面使用`with_config({})`方法修改配置中的`owner_repo_commit`字段，指定拉取`"rlm/rag-prompt-llama"`这个`commit`的`prompt`。然后使用新的配置，拉取更新后的`prompt`文件生成`prompt`。

In [12]:
prompt.with_config(configurable={"hub_commit": "rlm/rag-prompt-llama"})

RunnableBinding(bound=RunnableConfigurableFields(default=HubRunnable(bound=ChatPromptTemplate(input_variables=['context', 'question'], metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"))]), owner_repo_commit='rlm/rag-prompt'), fields={'owner_repo_commit': ConfigurableField(id='hub_commit', name='Hub Commit', description='The Hub commit to pull from', annotation=None, is_shared=False)}), config={'configurable': {'hub_commit': 'rlm/rag-prompt-llama'}})

In [ ]:
prompt.invoke(
    {"question": "foo", "context": "bar"}
)


总结起来，整个流程是：

- `configurable_fields` 声明可配置字段，定义其id为`'hub_commit'`，这个id就成了这个字段的唯一标识符
- 在 `with_config` 方法中，我们通过这个id `"hub_commit"` 来查找并更新相应的字段配置
最后使用更新后的配置生成`prompt`

  这个机制让整个自定义配置过程变得简洁高效,不需要指定配置在代码的哪个位置,只依赖 id 即可。


## Configurable Alternatives
### 配置不同的LLM



In [15]:
from langchain_community.chat_models import ChatAnthropic, ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import ConfigurableField

llm = ChatAnthropic(temperature=0).configurable_alternatives(
    # 给这个字段一个id，当配置最终的Runnable时,可以用这个id来配置这个字段
    ConfigurableField(id="llm"),
    # 设置一个默认键值，如果指定这个键值,将使用默认的LLM(上面初始化的ChatAnthropic)
    default_key="anthropic",
    # 添加一个名为`openai`的新选项,对应ChatOpenAI()
    openai=ChatOpenAI(),
    # 添加一个名为`gpt4`的新选项,对应ChatOpenAI(model="gpt-4")
    gpt4=ChatOpenAI(model="gpt-4"),  
    # 此处还可以添加更多的配置选项    
)

prompt = PromptTemplate.from_template("Tell me a joke about {topic}")

chain = prompt | llm

# By default it will call Anthropic
chain.invoke({"topic": "bears"})



d:\Users\DELL\miniconda3\envs\langchain\lib\site-packages\langchain_core\_api\deprecation.py:117: LangChainDeprecationWarning: The class `langchain_community.chat_models.openai.ChatOpenAI` was deprecated in langchain-community 0.0.10 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Claude API. Please go to Plans & Billing to upgrade or purchase credits.'}}

上述代码中，我们首先定义一个`ChatAnthropic`模型，设置`temperature`为0。然后：

- 使用`configurable_alternatives`方法，使这个`chat`模型成为可配置的。
- `ConfigurableField`方法给这个字段一个`id`，用于后续配置识别
- `default_key`参数设置默认key为"`anthropic`"（`ChatAnthropic`对应的key）
- 添加名为`openai`和`gpt4`的两个可选配置，分别对应不同的`ChatOpenAI`模型
- 构建`prompt`和`llm`的执行链`chain`


在这个`chain`中，默认情况下使用`ChatAnthropic`生成回复，因为`default_key="anthropic"`。我们可以使用`with_config`方法，通过`id "llm"`来切换`llm`为`ChatOpenAI`：

In [ ]:
# We can use `.with_config(configurable={"llm": "openai"})` to specify an llm to use
chain.with_config(configurable={"llm": "openai"}).invoke({"topic": "bears"})




将`"llm"`重新设置为`"anthropic"`，又恢复到默认的`ChatAnthropic`：


In [ ]:
# If we use the `default_key` then it uses the default
chain.with_config(configurable={"llm": "anthropic"}).invoke({"topic": "bears"})


## 配置不同的prompt
我们可以使用同样的方式配置不同的prompt：

In [ ]:
prompt = PromptTemplate.from_template(
    "Tell me a joke about {topic}"
).configurable_alternatives(
    # 给这个字段一个id，当配置最终的Runnable时,可以用这个id来配置这个字段
    ConfigurableField(id="prompt"),
    # 设置一个默认键值，如果指定这个键,将使用默认的Prompt模板(上面初始化的joke模板)
    default_key="joke",
    # 添加一个名为`poem`的新选项
    poem=PromptTemplate.from_template("Write a short poem about {topic}"),    
    # 此处可以添加更多的配置选项
)

chain = prompt | llm
# By default it will write a joke
chain.invoke({"topic": "bears"})


In [ ]:
## 也可以只更换一个配置项（llm）
chain.with_config(configurable={"llm": "openai"}).invoke({"topic": "bears"})

### 保存配置
我们也可以将配置后的`chain`保存为一个新的对象`（openai_poem）`，这样以后我们就可以通过调用`openai_poem`对象来使用这个特定的配置，而不需要每次都指定配置。

In [ ]:
openai_poem = chain.with_config(configurable={"llm": "openai"})
openai_poem.invoke({"topic": "bears"})